In [ ]:
"""ASVspoof5 training skeleton for handcrafted features.

Use this first for LFCC, then swap feature_name to mfcc/logmel/cqcc/plp.
Protocol files for spoof detection:
- ASVspoof5.train.tsv
- ASVspoof5.dev.track_1.tsv

Expected extracted folders (names may vary a bit):
- root/flac_T
- root/flac_D
- root/flac_E
- protocols/ASVspoof5.train.tsv
- protocols/ASVspoof5.dev.track_1.tsv
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Dataset

try:
    import torchaudio
    import torchaudio.transforms as T
except Exception as e:  # pragma: no cover
    torchaudio = None
    T = None
    TORCHAUDIO_IMPORT_ERROR = e
else:
    TORCHAUDIO_IMPORT_ERROR = None

import soundfile

print(torchaudio.list_audio_backends())
torchaudio.set_audio_backend("soundfile")

@dataclass
class Config:
    root_dir: str
    protocol_dir: str
    train_protocol_name: str = "ASVspoof5.train.tsv"
    dev_protocol_name: str = "ASVspoof5.dev.track_1.tsv"
    eval_protocol_name: Optional[str] = None
    train_audio_dir_hint: str = "flac_T"
    dev_audio_dir_hint: str = "flac_D"
    eval_audio_dir_hint: str = "flac_E"
    sample_rate: int = 16000
    batch_size: int = 32
    num_workers: int = 4
    lr: float = 1e-3
    epochs: int = 15
    hidden_dim: int = 256
    dropout: float = 0.3
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    max_audio_seconds: Optional[float] = None
    feature_name: str = "lfcc"
    pooling: str = "mean_std"
    n_mels: int = 40
    n_mfcc: int = 20
    n_lfcc: int = 20
    n_fft: int = 512
    win_length: int = 400
    hop_length: int = 160
    f_min: float = 0.0
    f_max: Optional[float] = None


def discover_audio_dir(root_dir: str, hint: str) -> Path:
    root = Path(root_dir)
    candidates = [root / hint, root / hint.lower(), root / hint.upper(), root / "audio" / hint, root / "data" / hint]
    for cand in candidates:
        if cand.exists():
            return cand
    matches = [p for p in root.rglob("*") if p.is_dir() and p.name == hint]
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Could not find directory '{hint}' under {root}")


def resolve_audio_path(audio_dir: Path, file_name: str, utt_id: str) -> Optional[Path]:
    candidates = [
        audio_dir / file_name,
        audio_dir / utt_id,
        audio_dir / f"{utt_id}.flac",
        audio_dir / f"{utt_id}.wav",
    ]
    for cand in candidates:
        if cand.exists() and cand.is_file():
            return cand
    matches = list(audio_dir.rglob(file_name))
    if matches:
        return matches[0]
    matches = list(audio_dir.rglob(f"{utt_id}.flac"))
    if matches:
        return matches[0]
    matches = list(audio_dir.rglob(f"{utt_id}.wav"))
    if matches:
        return matches[0]
    return None


def read_asvspoof5_protocol(protocol_path: str, split: str, audio_dir: str) -> pd.DataFrame:
    """Read ASVspoof5 train/dev track-1 TSV into a manifest."""
    path = Path(protocol_path)
    if not path.exists():
        raise FileNotFoundError(f"Missing protocol file: {path}")

    col_names = [
        "SPEAKER_ID",
        "FLAC_FILE_NAME",
        "SPEAKER_GENDER",
        "CODEC",
        "CODEC_Q",
        "CODEC_SEED",
        "ATTACK_TAG",
        "ATTACK_LABEL",
        "KEY",
        "TMP",
    ]

    df = pd.read_csv(
        path,
        sep=r"\s+",
        engine="python",
        header=None,
        names=col_names,
    )

    print(f"\n[DEBUG] Columns in {path}:")
    print(list(df.columns))
    print()

    records: List[Dict[str, str]] = []
    audio_root = Path(audio_dir)

    for _, row in df.iterrows():
        file_name = str(row["FLAC_FILE_NAME"]).strip()
        utt_id = Path(file_name).stem
        label_str = str(row["KEY"]).strip().lower()

        audio_path = resolve_audio_path(audio_root, file_name, utt_id)
        if audio_path is None:
            continue

        records.append(
            {
                "utt_id": utt_id,
                "file_name": file_name,
                "filepath": str(audio_path),
                "split": split,
                "label_str": label_str,
                "label": 1 if label_str == "spoof" else 0,
            }
        )

    if not records:
        raise RuntimeError(f"No usable rows found in {path}. Check paths and protocol parsing.")
    return pd.DataFrame.from_records(records)


# def build_manifests(cfg: Config) -> Dict[str, pd.DataFrame]:
#     protocol_dir = Path(cfg.protocol_dir)
#     train_protocol = protocol_dir / cfg.train_protocol_name
#     dev_protocol = protocol_dir / cfg.dev_protocol_name

#     train_audio_dir = discover_audio_dir(cfg.root_dir, cfg.train_audio_dir_hint)
#     dev_audio_dir = discover_audio_dir(cfg.root_dir, cfg.dev_audio_dir_hint)

#     manifests = {
#         "train": read_asvspoof5_protocol(str(train_protocol), "train", str(train_audio_dir)),
#         "dev": read_asvspoof5_protocol(str(dev_protocol), "dev", str(dev_audio_dir)),
#     }

#     if cfg.eval_protocol_name:
#         eval_protocol = protocol_dir / cfg.eval_protocol_name
#         eval_audio_dir = discover_audio_dir(cfg.root_dir, cfg.eval_audio_dir_hint)
#         manifests["eval"] = read_asvspoof5_protocol(str(eval_protocol), "eval", str(eval_audio_dir))

#     return manifests

def build_manifests(cfg: Config) -> Dict[str, pd.DataFrame]:
    protocol_dir = Path(cfg.protocol_dir)

    train_protocol = protocol_dir / cfg.train_protocol_name
    dev_protocol = protocol_dir / cfg.dev_protocol_name

    train_audio_dir = discover_audio_dir(cfg.root_dir, cfg.train_audio_dir_hint)
    dev_audio_dir = discover_audio_dir(cfg.root_dir, cfg.dev_audio_dir_hint)

    manifests = {
        "train": read_asvspoof5_protocol(str(train_protocol), "train", str(train_audio_dir)),
        "dev": read_asvspoof5_protocol(str(dev_protocol), "dev", str(dev_audio_dir)),
    }

    if cfg.eval_protocol_name:
        eval_protocol = protocol_dir / cfg.eval_protocol_name
        eval_audio_dir = discover_audio_dir(cfg.root_dir, cfg.eval_audio_dir_hint)
        manifests["eval"] = read_asvspoof5_protocol(str(eval_protocol), "eval", str(eval_audio_dir))

    return manifests

def save_manifest(df: pd.DataFrame, out_path: str) -> None:
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)


def load_audio(filepath: str, target_sr: int) -> torch.Tensor:
    if torchaudio is None:
        raise ImportError(f"torchaudio is required. Original import error: {TORCHAUDIO_IMPORT_ERROR}")
    wav, sr = torchaudio.load(filepath)
    if wav.ndim == 2 and wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != target_sr:
        wav = T.Resample(sr, target_sr)(wav)
    return wav.squeeze(0)


def maybe_crop_audio(wav: torch.Tensor, sr: int, max_seconds: Optional[float]) -> torch.Tensor:
    if max_seconds is None:
        return wav
    max_len = int(max_seconds * sr)
    return wav[:max_len] if wav.numel() > max_len else wav


class FeatureExtractor(nn.Module):
    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError


class MFCCExtractor(FeatureExtractor):
    def __init__(self, sample_rate: int, n_mfcc: int, n_mels: int, n_fft: int, win_length: int, hop_length: int):
        super().__init__()
        self.mfcc = T.MFCC(
            sample_rate=sample_rate,
            n_mfcc=n_mfcc,
            melkwargs={"n_fft": n_fft, "n_mels": n_mels, "win_length": win_length, "hop_length": hop_length, "center": True, "power": 2.0},
        )

    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        return self.mfcc(wav.unsqueeze(0)).squeeze(0)


class LogMelExtractor(FeatureExtractor):
    def __init__(self, sample_rate: int, n_mels: int, n_fft: int, win_length: int, hop_length: int):
        super().__init__()
        self.spec = T.MelSpectrogram(sample_rate=sample_rate, n_fft=n_fft, win_length=win_length, hop_length=hop_length, n_mels=n_mels, power=2.0)
        self.amplitude_to_db = T.AmplitudeToDB(stype="power")

    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        return self.amplitude_to_db(self.spec(wav.unsqueeze(0))).squeeze(0)


class LFCCExtractor(FeatureExtractor):
    def __init__(self, sample_rate: int, n_lfcc: int, n_fft: int, win_length: int, hop_length: int, f_min: float = 0.0, f_max: Optional[float] = None):
        super().__init__()
        self.sample_rate = sample_rate
        self.spectrogram = T.Spectrogram(n_fft=n_fft, win_length=win_length, hop_length=hop_length, power=2.0, center=True)
        self.n_lin = max(40, n_lfcc * 2)
        self.fbanks = self._build_linear_filterbanks(n_freqs=n_fft // 2 + 1, n_filters=self.n_lin, f_min=f_min, f_max=f_max or sample_rate / 2)
        self.dct_mat = self._build_dct(self.n_lin, n_lfcc)

    def _build_linear_filterbanks(self, n_freqs: int, n_filters: int, f_min: float, f_max: float) -> torch.Tensor:
        freqs = torch.linspace(0, self.sample_rate / 2, steps=n_freqs)
        edges = torch.linspace(f_min, f_max, steps=n_filters + 2)
        fb = torch.zeros(n_filters, n_freqs)
        for i in range(n_filters):
            left, center, right = edges[i], edges[i + 1], edges[i + 2]
            left_mask = (freqs >= left) & (freqs <= center)
            if center > left:
                fb[i, left_mask] = (freqs[left_mask] - left) / (center - left)
            right_mask = (freqs >= center) & (freqs <= right)
            if right > center:
                fb[i, right_mask] = (right - freqs[right_mask]) / (right - center)
        return fb / (fb.sum(dim=1, keepdim=True) + 1e-8)

    def _build_dct(self, n_filters: int, n_lfcc: int) -> torch.Tensor:
        basis = torch.zeros(n_lfcc, n_filters)
        scale = math.pi / n_filters
        for k in range(n_lfcc):
            for n in range(n_filters):
                basis[k, n] = math.cos((n + 0.5) * k * scale)
        basis[0] *= 1.0 / math.sqrt(2.0)
        return basis * math.sqrt(2.0 / n_filters)

    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        spec = self.spectrogram(wav.unsqueeze(0)).squeeze(0)
        fb = self.fbanks.to(spec.device, spec.dtype)
        dct = self.dct_mat.to(spec.device, spec.dtype)
        x = torch.matmul(fb, spec)
        x = torch.log(x + 1e-6)
        return torch.matmul(dct, x)


class CQCCExtractor(FeatureExtractor):
    def __init__(self, sample_rate: int, n_coeffs: int = 20):
        super().__init__()
        self.fallback = MFCCExtractor(sample_rate, n_coeffs, n_mels=40, n_fft=512, win_length=400, hop_length=160)

    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        return self.fallback(wav)


class PLPExtractor(FeatureExtractor):
    def __init__(self, sample_rate: int, n_coeffs: int = 13):
        super().__init__()
        self.fallback = MFCCExtractor(sample_rate, n_coeffs, n_mels=40, n_fft=512, win_length=400, hop_length=160)

    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        return self.fallback(wav)


def build_extractor(cfg: Config) -> FeatureExtractor:
    name = cfg.feature_name.lower()
    if name == "lfcc":
        return LFCCExtractor(cfg.sample_rate, cfg.n_lfcc, cfg.n_fft, cfg.win_length, cfg.hop_length, cfg.f_min, cfg.f_max)
    if name == "mfcc":
        return MFCCExtractor(cfg.sample_rate, cfg.n_mfcc, cfg.n_mels, cfg.n_fft, cfg.win_length, cfg.hop_length)
    if name == "logmel":
        return LogMelExtractor(cfg.sample_rate, cfg.n_mels, cfg.n_fft, cfg.win_length, cfg.hop_length)
    if name == "cqcc":
        return CQCCExtractor(cfg.sample_rate, cfg.n_mfcc)
    if name == "plp":
        return PLPExtractor(cfg.sample_rate, cfg.n_mfcc)
    raise ValueError(f"Unknown feature_name={cfg.feature_name}")


def pool_features(feat: torch.Tensor, mode: str = "mean_std") -> torch.Tensor:
    mean = feat.mean(dim=1)
    if mode == "mean":
        return mean
    if mode == "mean_std":
        return torch.cat([mean, feat.std(dim=1).clamp_min(1e-6)], dim=0)
    if mode == "mean_max":
        return torch.cat([mean, feat.max(dim=1).values], dim=0)
    raise ValueError(f"Unknown pooling mode: {mode}")


class SpoofDataset(Dataset):
    def __init__(self, df: pd.DataFrame, extractor: FeatureExtractor, sample_rate: int, max_audio_seconds: Optional[float], pooling: str):
        self.df = df.reset_index(drop=True)
        self.extractor = extractor
        self.sample_rate = sample_rate
        self.max_audio_seconds = max_audio_seconds
        self.pooling = pooling

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        wav = load_audio(row["filepath"], self.sample_rate)
        wav = maybe_crop_audio(wav, self.sample_rate, self.max_audio_seconds)
        feat = self.extractor(wav)
        x = pool_features(feat, self.pooling)
        y = torch.tensor(row["label"], dtype=torch.float32)
        return x, y, row["utt_id"]


class MLPClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


def compute_eer(y_true: np.ndarray, y_score: np.ndarray) -> float:
    fpr, tpr, _ = roc_curve(y_true, y_score)
    fnr = 1.0 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[idx] + fnr[idx]) / 2.0)


def evaluate(model: nn.Module, loader: DataLoader, device: str) -> Dict[str, float]:
    model.eval()
    all_y, all_prob = [], []
    with torch.no_grad():
        for x, y, _ in loader:
            x = x.to(device).float()
            prob = torch.sigmoid(model(x))
            all_y.append(y.numpy())
            all_prob.append(prob.cpu().numpy())
    y_true = np.concatenate(all_y)
    y_score = np.concatenate(all_prob)
    y_pred = (y_score >= 0.5).astype(np.int32)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "roc_auc": float(roc_auc_score(y_true, y_score)) if len(np.unique(y_true)) > 1 else float("nan"),
        "eer": compute_eer(y_true, y_score),
    }


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, device: str) -> float:
    model.train()
    total_loss = 0.0
    criterion = nn.BCEWithLogitsLoss()
    for x, y, _ in loader:
        x = x.to(device).float()
        y = y.to(device).float()
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / max(1, len(loader.dataset))


def infer_input_dim(loader: DataLoader) -> int:
    x, _, _ = next(iter(loader))
    return x.shape[1]


def make_loader(dataset: Dataset, batch_size: int, num_workers: int, shuffle: bool) -> DataLoader:
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers, pin_memory=torch.cuda.is_available(), drop_last=False)


def run_experiment(cfg: Config) -> Dict[str, float]:
    if torchaudio is None:
        raise ImportError(f"torchaudio is required. Original import error: {TORCHAUDIO_IMPORT_ERROR}")

    manifests = build_manifests(cfg)
    for split, df in manifests.items():
        save_manifest(df, f"{cfg.root_dir}/manifests/{split}.csv")

    extractor = build_extractor(cfg)
    train_ds = SpoofDataset(manifests["train"], extractor, cfg.sample_rate, cfg.max_audio_seconds, cfg.pooling)
    dev_ds = SpoofDataset(manifests["dev"], extractor, cfg.sample_rate, cfg.max_audio_seconds, cfg.pooling)
    train_loader = make_loader(train_ds, cfg.batch_size, cfg.num_workers, shuffle=True)
    dev_loader = make_loader(dev_ds, cfg.batch_size, cfg.num_workers, shuffle=False)

    input_dim = infer_input_dim(train_loader)
    model = MLPClassifier(input_dim=input_dim, hidden_dim=cfg.hidden_dim, dropout=cfg.dropout).to(cfg.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    best_state = None
    best_dev_eer = float("inf")
    for epoch in range(1, cfg.epochs + 1):
        loss = train_one_epoch(model, train_loader, optimizer, cfg.device)
        dev_metrics = evaluate(model, dev_loader, cfg.device)
        if dev_metrics["eer"] < best_dev_eer:
            best_dev_eer = dev_metrics["eer"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"Epoch {epoch:02d} | loss={loss:.4f} | dev_eer={dev_metrics['eer']:.4f} | dev_auc={dev_metrics['roc_auc']:.4f} | dev_acc={dev_metrics['accuracy']:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    final_dev = evaluate(model, dev_loader, cfg.device)
    print("Final dev metrics:", final_dev)

    results = {f"dev_{k}": v for k, v in final_dev.items()}
    if "eval" in manifests:
        eval_ds = SpoofDataset(manifests["eval"], extractor, cfg.sample_rate, cfg.max_audio_seconds, cfg.pooling)
        eval_loader = make_loader(eval_ds, cfg.batch_size, cfg.num_workers, shuffle=False)
        eval_metrics = evaluate(model, eval_loader, cfg.device)
        print("Eval metrics:", eval_metrics)
        results.update({f"eval_{k}": v for k, v in eval_metrics.items()})
    return results

In [ ]:
if __name__ == "__main__":
    cfg = Config(
        root_dir="/scratch1/kodachi/asvspoof5",
        protocol_dir="/scratch1/kodachi/asvspoof5",
        train_protocol_name="ASVspoof5.train.tsv",
        dev_protocol_name="ASVspoof5.dev.track_1.tsv",
        train_audio_dir_hint="flac_T",
        dev_audio_dir_hint="flac_D",
        feature_name="lfcc",
    )
    run_experiment(cfg)